# Homework 3

Solution to the third homework of the Machine Learning Zoomcamp 2026, in which I explore the lead-scoring dataset, compute the correlation and the mutual information between variables and train logistic regressions with different parameters

## Import Libraries and Load the Data

Import NumPy, pandas, seaborn, matplotlib and scikit-learn, load the dataset and inspect the first rows.

In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/refs/heads/main/cohorts/2026/data/course_lead_scoring_2026.csv")

In [3]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data Exploration

Inspect descriptive statistics, check missing values and fill the missing values of the categorical variables with 'NA' and of the numerical variables with '0.0'.

In [4]:
df.describe()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
count,4631.000000,5000.000000,5000.000000,4965.000000,5000.000000
mean,54533.212049,2.089400,4.472600,0.494767,0.576600
std,21245.661710,1.188987,2.435906,0.199431,0.494147
min,12000.000000,0.000000,0.000000,0.010000,0.000000
25%,40458.500000,1.000000,3.000000,0.360000,0.000000
50%,55051.000000,2.000000,4.000000,0.490000,1.000000
75%,71252.500000,3.000000,6.000000,0.630000,1.000000
max,109886.000000,6.000000,13.000000,0.990000,1.000000


In [5]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
df[['lead_source','industry','employment_status','location']] = df[['lead_source','industry','employment_status','location']].fillna('NA')

In [7]:
df[['annual_income','lead_score']] = df[['annual_income','lead_score']].fillna(0.0)

In [8]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

## Inspect the ```industry``` variable

Inspect the number of times each observation appears in the ```industry``` column and obtain the most frequent observation (mode) (Q1).

In [9]:
df.industry.value_counts()

technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: industry, dtype: int64

In [10]:
df.industry.mode()[0]

'technology'

## Correlation Matrix

Create the correlation matrix between the numerical features of the dataset (Q2).

In [11]:
numerical = ['annual_income','number_of_courses_viewed','interaction_count','lead_score']
categorical = ['lead_source','industry','employment_status','location']

In [12]:
df[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


The two features with the biggest correlation are ```interaction_count``` and ```lead_score```.

## Split the data

Split the data into train, validation and test datasets.

In [13]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [14]:
y_train = df_train.converted.values
y_val = df_val.converted.values

## Mutual Information Score

Calculate the mutual information score between ```converted``` and the other categorical variables of the dataset (Q3).

In [15]:
def calculate_mi(series):
    return mutual_info_score(series, df_train.converted)


df_mi = round(df_train[categorical].apply(calculate_mi),2)
df_mi = df_mi.sort_values(ascending=False).to_frame(name='MI')

df_mi

,MI
lead_source,0.03
employment_status,0.02
industry,0.00
location,0.00


The variable with the biggest mutual information score is ```lead_source```.

In [16]:
del df_train['converted']
del df_val['converted']

## Logistic Regression

Train a logistic regression, fit the model on the training dataset and calculate the accuracy on the validation dataset (Q4).

In [17]:
train_dict = df_train[categorical + numerical].to_dict(orient='records')

In [18]:
train_dict[0]

{'lead_source': 'paid_ads',
 'industry': 'finance',
 'employment_status': 'employed',
 'location': 'asia',
 'annual_income': 65980.0,
 'number_of_courses_viewed': 2,
 'interaction_count': 3,
 'lead_score': 0.37}

In [19]:
dv = DictVectorizer(sparse=False)
dv.fit(train_dict)

DictVectorizer(sparse=False)

In [20]:
X_train = dv.transform(train_dict)

In [21]:
X_train.shape

(3000, 28)

In [22]:
dv.get_feature_names_out()

array(['annual_income', 'employment_status=NA',
       'employment_status=employed', 'employment_status=self_employed',
       'employment_status=student', 'employment_status=unemployed',
       'industry=NA', 'industry=education', 'industry=finance',
       'industry=healthcare', 'industry=manufacturing', 'industry=retail',
       'industry=technology', 'interaction_count', 'lead_score',
       'lead_source=NA', 'lead_source=events',
       'lead_source=organic_search', 'lead_source=paid_ads',
       'lead_source=referral', 'lead_source=social_media', 'location=NA',
       'location=africa', 'location=asia', 'location=europe',
       'location=north_america', 'location=south_america',
       'number_of_courses_viewed'], dtype=object)

In [23]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [24]:
val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [25]:
y_pred = model.predict_proba(X_val)[:, 1]

In [26]:
converted = y_pred > 0.5

In [27]:
accuracy_full = (y_val == converted).mean()

print(f"{accuracy_full:.2f}")

0.65


The accuracy of the model on the validation dataset is 0.65.

## Feature Elimination

Train the same logistic regression model as before but excluding one feature at a time. Record the accuracy for each model. For each model, calculate the difference between the original accuracy and the accuracy without the feature (Q5).

In [28]:
features = categorical + numerical
partial_accuracies = {}

for i in range(len(features)):
    other_features = features[:i] + features[i+1:]

    train_dict = df_train[other_features].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    dv.fit(train_dict)

    X_train = dv.transform(train_dict)

    model.fit(X_train, y_train)

    val_dict = df_val[other_features].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    y_pred = model.predict_proba(X_val)[:, 1]

    converted = y_pred > 0.5

    partial_accuracies[features[i]] = (y_val == converted).mean()

partial_accuracies

{'lead_source': 0.642,
 'industry': 0.645,
 'employment_status': 0.645,
 'location': 0.645,
 'annual_income': 0.724,
 'number_of_courses_viewed': 0.643,
 'interaction_count': 0.601,
 'lead_score': 0.644}

In [29]:
accuracy_differences = {}

for i in partial_accuracies:
    accuracy_differences[i] = accuracy_full - partial_accuracies[i]

accuracy_differences    

{'lead_source': 0.0030000000000000027,
 'industry': 0.0,
 'employment_status': 0.0,
 'location': 0.0,
 'annual_income': -0.07899999999999996,
 'number_of_courses_viewed': 0.0020000000000000018,
 'interaction_count': 0.04400000000000004,
 'lead_score': 0.0010000000000000009}

The feature ```number_of_courses_viewed``` has a smaller difference than the features ```lead_source``` and ```interaction_count```.

## Regularized Logistic Regression

Train a regularized logistic regression for different values of the regularization parameter ```C```. Compare the accuracies on the validation dataset obtained for the different values of ```C``` (Q6).

In [30]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]

accuracies_C = {}

train_dict = df_train[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
dv.fit(train_dict)

X_train = dv.transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

for c in C_values:
    
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)

    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]

    converted = y_pred > 0.5

    accuracies_C[c] = round((y_val == converted).mean(),3)

accuracies_C

{1e-06: 0.598, 1e-05: 0.598, 0.0001: 0.613, 0.001: 0.645}

```C = 0.001``` leads to the best accuracy on the validation dataset.